<a href="https://colab.research.google.com/github/natalianowak1/airbnb-price-optimization/blob/main/airbnb_price_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Przygotowanie środowiska i import bibliotek



In [ ]:
import pandas as pd

# 2. Wczytanie i wstępny przegląd danych

In [ ]:
df = pd.read_excel("/content/Airbnb_Open_Data.xlsx")

Podgląd pierwszych pięciu wierszy

In [ ]:
pd.set_option('display.max_columns', None)
print(df.head())

Identyfikacja zmiennych i weryfikacja typów

In [ ]:
print(df.info())

Zliczanie brakujących wartości (nulli) w każdej kolumnie

In [ ]:
print(df.isnull().sum())

Liczba zduplikowanych wierszy

In [ ]:
print(df.duplicated().sum())

Wnioski ze wstępnej analizy:
1. Metryki ogólne:
*  Tabela zawiera 25 kolumnn o nazwach kolejno: 'id', 'NAME', 'host id', 'host_identity_verified', 'host name', 'neighbourhood group', 'neighbourhood', 'latitude', 'longitude', 'country', 'country code', 'instant_bookable', 'cancellation_policy', 'room type', 'Construction year', 'price', 'service fee', 'minimum nights', 'number of reviews', 'reviews per month', 'review rate number', 'calculated host listings count', 'availability 365', 'house_rules', 'license'
* Tabela liczy 102 599 wierszy (ogłoszeń).
* W zbiorze wykryto 541 całkowicie zduplikowanych wierszy, które należy usunąć.
* Jedyne kolumny, które są w 100% kompletne (0 nulli), to: `id`, `host id` oraz `room type`. W pozostałych 22 kolumnach występują nulle.



2. Zidentyfikowane anomalie i plan ich czyszczenia:
* Zbiór danych charakteryzuje się brakiem spójności w nazwach kolumn (część pisana wielkimi literami, stosowanie spacji na zmianę z podkreśleniami/podłogami). Aby uprosćić kod w Pythonie (zamiast konieczności pisania "df['host id']", można pisać: "df.host_id") oraz zapewnić spójność, nazwy kolumn należy zamienić do formatu `pierwszesłowo_drugiesłowo` (małe litery, spacje zastąpione znakiem `_`).
*   Kolumny takie jak 'price' (cena) oraz 'service fee' (opłata serwisowa) są już zapisane jako dane liczbowe (float64), co umożliwia bezpośrednie wykonywanie obliczeń i analizę statystyczną. Posiadają jednak po 247 braków danych, które trzeba uzupełnić (np. medianą cen).
* Kolumna `instant_bookable` zawiera wartości binarne (1 i 0) oraz 105 nulli. W etapie czyszczenia należy zastąpić nulle wartością `0` (bezpieczne założenie, że brak informacji oznacza brak natychmiastowej rezerwacji), a całą kolumnę zmienić na typ logiczny (`boolean`).
* Kolumna `reviews per month` zawiera aż 15 879 braków danych. Wynika to z faktu, że nowo dodane nieruchomości nie mają jeszcze żadnych opinii. Nulle w tej kolumnie zostaną zastąpione wartością `0.0`.
* Kolumna 'license' zawiera tylko 2 wypełnione wiersze, a cała reszta to nulle (kolumna kwalifikuje się do usunięcia).
* Kolumna `house_rules` zawiera informacje w mniej niż połowie wierszy (52 131 nulli). Braki zostaną zastąpione wartością domyślną "No rules specified".
* Należy wykonać korektę kolumn lokalizacyjnych (`country` i `country code`) Ponieważ cały zbiór danych dotyczy rynku w Nowym Jorku, braki danych w kolumnie kraju i jego kodu zostaną uzupełnione stałymi wartościami (odpowiednio: "United States" oraz "US")
* Kolumna Zależność weryfikacji hosta (`host_identity_verified`) posiada 289 nulli. Zostanie tam wprowadzona kategoria "unconfirmed", co jest bezpieczniejszym podejściem biznesowym niż zakładanie, że profil jest zweryfikowany
* Wykryto po 8 braków danych w pozycjach geograficznych (`latitude`, `longitude`). Ponieważ te współrzędne są kluczowe do poprawnego renderowania map w Tableau, wiersze z tymi brakami zostaną całkowicie usunięte z bazy.
* Pozostałe kolumny tekstowe i lokalizacyjne (np. `NAME`, `host name`, `neighbourhood`) posiadają niewielkie liczby braków (od kilku do kilkuset). W ich przypadku wiersze z nullami zostaną zastąpione wartością "Unknown".



# 3. Czyszczenie danych (Data Cleaning)


Usuwanie całkowicie zduplikowanych wierszy

In [ ]:
print(f"Liczba wierszy przed usunięciem duplikatów: {len(df)}")
df = df.drop_duplicates()
print(f"Liczba wierszy po usunięciu duplikatów: {len(df)}\n")

Usuwanie wierszy, które mają nulle w latitude lub longitude

In [ ]:
print(f"Liczba wierszy przed usunięciem braków w geolokalizacji: {len(df)}")
df = df[(df['latitude'].notnull()) & (df['longitude'].notnull())]
print(f"Liczba wierszy po usunięciu braków w geolokalizacji: {len(df)}")

Usuwanie kolumny `license`

In [ ]:
df = df.drop(columns=['license'])
print(f"Nowa liczba kolumn w tabeli: {len(df.columns)}")

Zmiana nazw kolumn do formatu "snake_case"

In [ ]:
print(list(df.columns))
#zamiana liter z wielkich ma małe
df.columns = df.columns.str.lower()
#zamiana spacji w nazwach kolumn na podkreśclenie
df.columns = df.columns.str.lower().str.replace(' ', '_')
print(list(df.columns))

Uzupełnienie Nulli zerami w kolumnie `instant_bookable`

In [ ]:
print(f"Liczba nulli w instant_bookable przed uzupełnieniem zerami: {df['instant_bookable'].isnull().sum()}")
df['instant_bookable'] = df['instant_bookable'].fillna(0)
print(f"Liczba nulli w instant_bookable: {df.instant_bookable.isnull().sum()}")

Zmieniamy typ kolumny `instant_bookable` na boolean

In [ ]:
print(f"Typ kolumny przed zmianą: {df.instant_bookable.dtype}")
df['instant_bookable'] = df['instant_bookable'].astype(bool)
print(f"Nowy typ kolumny: {df.instant_bookable.dtype}")

Uzupełnienie nulli w recenzjach na miesiąc (`reviews_per_month`) zerem

In [ ]:
print(f"Liczba nulli w reviews per month przed uzupełnieniem: {df.reviews_per_month.isnull().sum()}")
df['reviews_per_month'] = df['reviews_per_month'].fillna(0.0)
print(f"Liczba nulli w reviews per month: {df.reviews_per_month.isnull().sum()}")

Zastępowanie braków tekstem domyślnym ("No rules specified") w kolumnie `house_rules`

In [ ]:
print(f"Liczba nulli w house_rules przed uzupełnieniem: {df.house_rules.isnull().sum()}\n")
df['house_rules'] = df['house_rules'].fillna("No rules specified")
print(f"Liczba nulli w house_rules: {df.house_rules.isnull().sum()}\n")

Uzupełnianie stałą wartością kolumny `country`

In [ ]:
print(f"Liczba nulli w country przed uzupełnieniem: {df.country.isnull().sum()}")
df['country'] = df['country'].fillna("United States")
print(f"Liczba nulli w country: {df.country.isnull().sum()}")

Uzupełnianie stałą wartością kolumny `country_code`

In [ ]:
print(f"Liczba nulli w country_code przed uzupełnieniem: {df.country_code.isnull().sum()}")
df['country_code'] = df['country_code'].fillna("US")
print(f"Liczba nulli w country_code: {df.country_code.isnull().sum()}")

Weryfikacja gospodarza `host_identity_verified`: uzupełnienie nulli wartością "unconfirmed"

In [ ]:
print(f"Liczba nulli w host_identity_verified przed uzupełnieniem: {df.host_identity_verified.isnull().sum()}\n")
df['host_identity_verified'] = df['host_identity_verified'].fillna("unconfirmed")
print(f"Liczba nulli w host_identity_verified: {df.host_identity_verified.isnull().sum()}")

Zastępowanie braków w kolumnach `name` i `host_name` wartością "Unknown"

In [ ]:
print(f"Liczba nulli w name przed uzupełnieniem: {df.name.isnull().sum()}")
print(f"Liczba nulli w host_name przed uzupełnieniem: {df.host_name.isnull().sum()}\n")
df['name'] = df['name'].fillna("Unknown")
df['host_name'] = df['host_name'].fillna("Unknown")
print(f"Liczba nulli w name: {df.name.isnull().sum()}")
print(f"Liczba nulli w host_name: {df.host_name.isnull().sum()}")